# Evaluating Model Outputs

We can evaluate a model's confidence in its results by using perplexity. Perplexity is a measure of uncertainty that can be calculated by exponentiating the negative of the average of the logprobs. 

+ Perplexity can be used to assess the result of an individual model run.
+ It can also be used to compare the relative confidence of results between model runs. 

Low perplexity or high confidence does not guarantee accuracy, but it can be a helpful signal when paired with other evaluation metrics. 

In [1]:
%load_ext dotenv
%dotenv ../../05_src/.env
%dotenv ../../05_src/.secrets
import sys
sys.path.append('../../05_src/')

In [2]:
from utils.clients import get_client
from IPython.display import display, Markdown
import numpy as np

client = get_client()

In [3]:
prompts = [
    # Low perplexity: Clear topic, common structure, highly predictable vocabulary
    "Explain how photosynthesis works in simple terms.",
    # Medium preplexity: Narrative freedom, but familiar theme and constraints.
    "Write a short story about a traveler who realizes the journey mattered more than the destination.",
    # High perplexity: Abstract concept, creative freedom, unpredictable vocabulary
    "Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather."
]

In [4]:
def get_completion(
    input: list[dict[str, str]],
    model: str = "gpt-4o-mini",
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  # whether to return log probabilities of the output tokens or not. If true, returns the log probabilities of each output token returned in the content of message..
    top_logprobs=None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        "tools": tools,
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
    }
    if tools:
        params["tools"] = tools

    completion = client.responses.create(**params)
    return completion

In [10]:
for k, prompt in enumerate(prompts):
    API_RESPONSE = get_completion(
        [{"role": "user", "content": prompt}],
        model="gpt-4o-mini",
        logprobs=True,
    )
    token_data = API_RESPONSE.output[0].content[0].logprobs
    response_text = API_RESPONSE.output[0].content[0].text
    lp_values = [t.logprob for t in token_data]
    perplexity_score = np.exp(-np.mean(lp_values))

    rows = [
        f"| `{t.token.replace('|', chr(9474)).replace(chr(10), '↵').replace(chr(96), chr(39))}` "
        f"| {t.logprob:.4f} | {np.exp(t.logprob)*100:.1f}% |"
        for t in token_data
    ]
    table = "\n".join([
        "| Token | Logprob | Linear Prob |",
        "|:------|--------:|------------:|",
    ] + rows)

    display(Markdown(f"### Prompt {k+1}\n_{prompt}_"))
    display(Markdown(f"**Response:**\n\n{response_text}"))
    display(Markdown(table))
    display(Markdown(f"**Perplexity:** `{perplexity_score:.2f}`\n\n---"))

### Prompt 1
_Explain how photosynthesis works in simple terms._

**Response:**

Photosynthesis is the process that plants, algae, and some bacteria use to make their own food. Here’s how it works in simple terms:

1. **Sunlight**: Plants take in sunlight using a green pigment called chlorophyll, which is found in their leaves.

2. **Water**: Plants absorb water from the soil through their roots.

3. **Carbon Dioxide**: Plants take in carbon dioxide from the air through tiny openings in their leaves called stomata.

4. **Making Food**: Using the energy from sunlight, plants combine water and carbon dioxide to create glucose (a type of sugar) and oxygen. The glucose is used as food for energy and growth.

5. **Oxygen Release**: The oxygen produced during this process is released back into the air, which is essential for us and other living beings to breathe.

In summary, photosynthesis is how plants turn sunlight, water, and carbon dioxide into food and oxygen!

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Photos` | -0.0623 | 94.0% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -0.0000 | 100.0% |
| ` the` | -0.0428 | 95.8% |
| ` process` | -0.0041 | 99.6% |
| ` that` | -0.2843 | 75.3% |
| ` plants` | -0.0047 | 99.5% |
| `,` | -0.5766 | 56.2% |
| ` algae` | -0.0289 | 97.2% |
| `,` | 0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` some` | -0.0000 | 100.0% |
| ` bacteria` | -0.0001 | 100.0% |
| ` use` | -0.0000 | 100.0% |
| ` to` | 0.0000 | 100.0% |
| ` make` | -0.1557 | 85.6% |
| ` their` | -0.0067 | 99.3% |
| ` own` | -0.0789 | 92.4% |
| ` food` | -0.0000 | 100.0% |
| `.` | -0.5786 | 56.1% |
| ` Here` | -0.1630 | 85.0% |
| `’s` | -0.0001 | 100.0% |
| ` how` | -0.1002 | 90.5% |
| ` it` | 0.0000 | 100.0% |
| ` works` | -0.0000 | 100.0% |
| ` in` | -0.0238 | 97.6% |
| ` simple` | -0.0001 | 100.0% |
| ` terms` | -0.0015 | 99.8% |
| `:↵↵` | -0.0000 | 100.0% |
| `1` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Sun` | -0.7171 | 48.8% |
| `light` | -0.0003 | 100.0% |
| `**` | -0.0996 | 90.5% |
| `:` | -0.0000 | 100.0% |
| ` Plants` | -0.0010 | 99.9% |
| ` take` | -0.7736 | 46.1% |
| ` in` | -0.0007 | 99.9% |
| ` sunlight` | -0.0190 | 98.1% |
| ` using` | -0.5016 | 60.6% |
| ` a` | -0.2050 | 81.5% |
| ` green` | -0.0320 | 96.9% |
| ` pigment` | -0.0021 | 99.8% |
| ` called` | -0.0298 | 97.1% |
| ` chlor` | -0.0000 | 100.0% |
| `ophyll` | 0.0000 | 100.0% |
| `,` | -0.5952 | 55.1% |
| ` which` | -0.7341 | 48.0% |
| ` is` | -0.0002 | 100.0% |
| ` found` | -0.4199 | 65.7% |
| ` in` | -0.0135 | 98.7% |
| ` their` | -0.0027 | 99.7% |
| ` leaves` | -0.0000 | 100.0% |
| `.↵↵` | -0.0034 | 99.7% |
| `2` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Water` | -0.3352 | 71.5% |
| `**` | -0.3155 | 72.9% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.6427 | 52.6% |
| ` absorb` | -0.0056 | 99.4% |
| ` water` | -0.0000 | 100.0% |
| ` from` | -0.0314 | 96.9% |
| ` the` | -0.0001 | 100.0% |
| ` soil` | -0.0142 | 98.6% |
| ` through` | -0.0000 | 100.0% |
| ` their` | -0.0000 | 100.0% |
| ` roots` | -0.0000 | 100.0% |
| `.↵↵` | -0.0006 | 99.9% |
| `3` | 0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Carbon` | -0.0005 | 100.0% |
| ` D` | -0.0067 | 99.3% |
| `ioxide` | -0.0000 | 100.0% |
| `**` | -0.0002 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` Plants` | -0.0533 | 94.8% |
| ` take` | -0.0365 | 96.4% |
| ` in` | -0.0005 | 100.0% |
| ` carbon` | -0.0042 | 99.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` from` | -0.0551 | 94.6% |
| ` the` | -0.0000 | 100.0% |
| ` air` | -0.0000 | 100.0% |
| ` through` | -0.0002 | 100.0% |
| ` tiny` | -0.0790 | 92.4% |
| ` openings` | -0.0113 | 98.9% |
| ` in` | -0.0189 | 98.1% |
| ` their` | -0.0003 | 100.0% |
| ` leaves` | 0.0000 | 100.0% |
| ` called` | -0.0070 | 99.3% |
| ` stom` | -0.0002 | 100.0% |
| `ata` | -0.0000 | 100.0% |
| `.↵↵` | -0.0000 | 100.0% |
| `4` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `Making` | -1.1455 | 31.8% |
| ` Food` | -0.0044 | 99.6% |
| `**` | -0.0001 | 100.0% |
| `:` | -0.0001 | 100.0% |
| ` Using` | -0.0545 | 94.7% |
| ` the` | -0.4562 | 63.4% |
| ` energy` | -0.2529 | 77.7% |
| ` from` | -0.0000 | 100.0% |
| ` sunlight` | -0.0032 | 99.7% |
| `,` | -0.0000 | 100.0% |
| ` plants` | -0.0097 | 99.0% |
| ` combine` | -0.1541 | 85.7% |
| ` water` | -0.2019 | 81.7% |
| ` and` | -0.0001 | 100.0% |
| ` carbon` | 0.0000 | 100.0% |
| ` dioxide` | 0.0000 | 100.0% |
| ` to` | -0.0036 | 99.6% |
| ` create` | -0.3005 | 74.0% |
| ` glucose` | -0.0219 | 97.8% |
| ` (` | -0.0382 | 96.3% |
| `a` | -0.0004 | 100.0% |
| ` type` | -0.0101 | 99.0% |
| ` of` | 0.0000 | 100.0% |
| ` sugar` | -0.0000 | 100.0% |
| `)` | -0.0998 | 90.5% |
| ` and` | -0.0021 | 99.8% |
| ` oxygen` | -0.0006 | 99.9% |
| `.` | -0.1604 | 85.2% |
| ` The` | -0.7192 | 48.7% |
| ` glucose` | -0.4097 | 66.4% |
| ` is` | -0.6193 | 53.8% |
| ` used` | -0.4672 | 62.7% |
| ` as` | -0.2396 | 78.7% |
| ` food` | -0.0233 | 97.7% |
| ` for` | -0.0440 | 95.7% |
| ` energy` | -0.4985 | 60.7% |
| ` and` | -0.0187 | 98.1% |
| ` growth` | -0.0000 | 100.0% |
| `.↵↵` | -0.3137 | 73.1% |
| `5` | -0.0000 | 100.0% |
| `.` | 0.0000 | 100.0% |
| ` **` | 0.0000 | 100.0% |
| `O` | -0.1491 | 86.1% |
| `xygen` | -0.0000 | 100.0% |
| ` Release` | -0.0371 | 96.4% |
| `**` | -0.0000 | 100.0% |
| `:` | 0.0000 | 100.0% |
| ` The` | -0.7224 | 48.6% |
| ` oxygen` | -0.1192 | 88.8% |
| ` produced` | -0.0272 | 97.3% |
| ` during` | -0.1473 | 86.3% |
| ` this` | -0.0407 | 96.0% |
| ` process` | -0.0000 | 100.0% |
| ` is` | -0.0001 | 100.0% |
| ` released` | -0.0024 | 99.8% |
| ` back` | -0.5759 | 56.2% |
| ` into` | 0.0000 | 100.0% |
| ` the` | 0.0000 | 100.0% |
| ` air` | -0.0142 | 98.6% |
| `,` | -0.0043 | 99.6% |
| ` which` | -0.0018 | 99.8% |
| ` is` | -0.0183 | 98.2% |
| ` essential` | -1.0040 | 36.6% |
| ` for` | 0.0000 | 100.0% |
| ` us` | -0.8869 | 41.2% |
| ` and` | -0.1029 | 90.2% |
| ` other` | -0.0638 | 93.8% |
| ` living` | -0.1030 | 90.2% |
| ` beings` | -0.5460 | 57.9% |
| ` to` | -0.0299 | 97.1% |
| ` breathe` | -0.0000 | 100.0% |
| `.↵↵` | -0.0007 | 99.9% |
| `In` | -0.3878 | 67.9% |
| ` summary` | -0.1640 | 84.9% |
| `,` | -0.0047 | 99.5% |
| ` photos` | -0.0862 | 91.7% |
| `ynthesis` | 0.0000 | 100.0% |
| ` is` | -1.0384 | 35.4% |
| ` how` | -0.0645 | 93.8% |
| ` plants` | -0.0001 | 100.0% |
| ` turn` | -0.3721 | 68.9% |
| ` sunlight` | -0.0026 | 99.7% |
| `,` | -0.0298 | 97.1% |
| ` water` | -0.0002 | 100.0% |
| `,` | -0.0000 | 100.0% |
| ` and` | 0.0000 | 100.0% |
| ` carbon` | -0.0142 | 98.6% |
| ` dioxide` | 0.0000 | 100.0% |
| ` into` | 0.0000 | 100.0% |
| ` food` | -0.0059 | 99.4% |
| ` and` | -0.1331 | 87.5% |
| ` oxygen` | -0.0010 | 99.9% |
| `!` | -0.2167 | 80.5% |

**Perplexity:** `1.12`

---

### Prompt 2
_Write a short story about a traveler who realizes the journey mattered more than the destination._

...

**Perplexity:** `1.50`

---

### Prompt 3
_Describe the taste of a color that only exists for one second at dusk, using metaphors from mathematics and weather._

**Response:**

Imagine a color that tastes like the fleeting moment when twilight kisses the horizon—a blend of soft lavender and deep indigo, like the gentle curve of a parabolic arc just before it meets the ground. It’s the taste of a cool breeze, whispering secrets of a storm that never arrives, a fleeting probability that dances on the edge of certainty.

This color tastes like the sweet tang of dew on grass, a fleeting moment of clarity in a chaotic equation, where the variables align just right. It’s the ephemeral flavor of a gentle rain, each drop a note in a symphony of potential, harmonizing with the fading light. 

As it lingers for just a second, it’s like the last digit of pi—infinitely complex yet tantalizingly elusive, leaving a trace of warmth that feels like the sun’s final embrace before surrendering to the night.

| Token | Logprob | Linear Prob |
|:------|--------:|------------:|
| `Imagine` | -0.3913 | 67.6% |
| ` a` | -0.5276 | 59.0% |
| ` color` | -0.4897 | 61.3% |
| ` that` | -0.0090 | 99.1% |
| ` tastes` | -1.4523 | 23.4% |
| ` like` | -0.0004 | 100.0% |
| ` the` | -0.2201 | 80.2% |
| ` fleeting` | -0.1831 | 83.3% |
| ` moment` | -0.3248 | 72.3% |
| ` when` | -0.2129 | 80.8% |
| ` twilight` | -0.7537 | 47.1% |
| ` kisses` | -1.8867 | 15.2% |
| ` the` | -0.0288 | 97.2% |
| ` horizon` | -0.0899 | 91.4% |
| `—a` | -0.8063 | 44.7% |
| ` blend` | -1.6832 | 18.6% |
| ` of` | -0.0266 | 97.4% |
| ` soft` | -1.6512 | 19.2% |
| ` lavender` | -0.6170 | 54.0% |
| ` and` | -0.0123 | 98.8% |
| ` deep` | -0.2927 | 74.6% |
| ` ind` | -0.1428 | 86.7% |
| `igo` | -0.0000 | 100.0% |
| `,` | -0.3577 | 69.9% |
| ` like` | -1.0757 | 34.1% |
| ` the` | -0.3788 | 68.5% |
| ` gentle` | -0.6114 | 54.3% |
| ` curve` | -0.3973 | 67.2% |
| ` of` | -0.0000 | 100.0% |
| ` a` | -0.0431 | 95.8% |
| ` par` | -0.6071 | 54.5% |
| `abolic` | -0.0014 | 99.9% |
| ` arc` | -0.0506 | 95.1% |
| ` just` | -1.3514 | 25.9% |
| ` before` | -0.0140 | 98.6% |
| ` it` | -0.0147 | 98.5% |
| ` meets` | -1.5007 | 22.3% |
| ` the` | -0.2791 | 75.6% |
| ` ground` | -0.5281 | 59.0% |
| `.` | -0.0404 | 96.0% |
| ` It` | -0.7898 | 45.4% |
| `’s` | -0.4682 | 62.6% |
| ` the` | -0.6238 | 53.6% |
| ` taste` | -1.3299 | 26.5% |
| ` of` | -0.0004 | 100.0% |
| ` a` | -0.2641 | 76.8% |
| ` cool` | -0.7899 | 45.4% |
| ` breeze` | -0.0453 | 95.6% |
| `,` | -1.3491 | 25.9% |
| ` whisper` | -1.2759 | 27.9% |
| `ing` | -0.0001 | 100.0% |
| ` secrets` | -0.6919 | 50.1% |
| ` of` | -0.1717 | 84.2% |
| ` a` | -1.8623 | 15.5% |
| ` storm` | -1.4418 | 23.6% |
| ` that` | -0.2649 | 76.7% |
| ` never` | -0.2564 | 77.4% |
| ` arrives` | -0.9709 | 37.9% |
| `,` | -0.0666 | 93.6% |
| ` a` | -1.4179 | 24.2% |
| ` fleeting` | -1.6832 | 18.6% |
| ` probability` | -1.8194 | 16.2% |
| ` that` | -0.4907 | 61.2% |
| ` dances` | -0.9050 | 40.5% |
| ` on` | -0.9883 | 37.2% |
| ` the` | -0.0016 | 99.8% |
| ` edge` | -0.0778 | 92.5% |
| ` of` | -0.0006 | 99.9% |
| ` certainty` | -0.2161 | 80.6% |
| `.↵↵` | -0.1499 | 86.1% |
| `This` | -0.1613 | 85.1% |
| ` color` | -0.3037 | 73.8% |
| ` tastes` | -0.9433 | 38.9% |
| ` like` | -0.0162 | 98.4% |
| ` the` | -0.2513 | 77.8% |
| ` sweet` | -2.0362 | 13.1% |
| ` tang` | -0.4039 | 66.8% |
| ` of` | -0.0000 | 100.0% |
| ` dew` | -1.5919 | 20.4% |
| ` on` | -0.9354 | 39.2% |
| ` grass` | -0.6342 | 53.0% |
| `,` | -0.6612 | 51.6% |
| ` a` | -1.5147 | 22.0% |
| ` fleeting` | -1.6815 | 18.6% |
| ` moment` | -1.5412 | 21.4% |
| ` of` | -1.1261 | 32.4% |
| ` clarity` | -0.9894 | 37.2% |
| ` in` | -0.8430 | 43.0% |
| ` a` | -0.3903 | 67.7% |
| ` chaotic` | -0.6374 | 52.9% |
| ` equation` | -0.2242 | 79.9% |
| `,` | -0.4487 | 63.8% |
| ` where` | -0.1592 | 85.3% |
| ` the` | -1.1634 | 31.2% |
| ` variables` | -0.3677 | 69.2% |
| ` align` | -0.2887 | 74.9% |
| ` just` | -0.6389 | 52.8% |
| ` right` | -1.0216 | 36.0% |
| `.` | -0.8528 | 42.6% |
| ` It` | -0.1477 | 86.3% |
| `’s` | -0.1302 | 87.8% |
| ` the` | -0.6588 | 51.7% |
| ` ephemeral` | -1.5594 | 21.0% |
| ` flavor` | -1.8273 | 16.1% |
| ` of` | -0.0012 | 99.9% |
| ` a` | -0.3386 | 71.3% |
| ` gentle` | -1.7273 | 17.8% |
| ` rain` | -0.4023 | 66.9% |
| `,` | -0.3876 | 67.9% |
| ` each` | -1.3111 | 27.0% |
| ` drop` | -0.1137 | 89.3% |
| ` a` | -0.0832 | 92.0% |
| ` note` | -1.6707 | 18.8% |
| ` in` | -0.0132 | 98.7% |
| ` a` | -0.0894 | 91.4% |
| ` sym` | -0.2834 | 75.3% |
| `phony` | -0.0430 | 95.8% |
| ` of` | -0.2171 | 80.5% |
| ` potential` | -2.4789 | 8.4% |
| `,` | -0.1382 | 87.1% |
| ` harmon` | -1.7366 | 17.6% |
| `izing` | -0.0008 | 99.9% |
| ` with` | -1.2986 | 27.3% |
| ` the` | -0.0067 | 99.3% |
| ` fading` | -1.4682 | 23.0% |
| ` light` | -0.5809 | 55.9% |
| `.` | -0.7717 | 46.2% |
| ` ↵↵` | -0.7863 | 45.6% |
| `As` | -0.3475 | 70.6% |
| ` it` | -0.4699 | 62.5% |
| ` l` | -1.4527 | 23.4% |
| `ingers` | -0.0000 | 100.0% |
| ` for` | -0.2497 | 77.9% |
| ` just` | -0.6910 | 50.1% |
| ` a` | -0.5553 | 57.4% |
| ` second` | -0.4488 | 63.8% |
| `,` | -0.0150 | 98.5% |
| ` it` | -0.1351 | 87.4% |
| `’s` | -0.6448 | 52.5% |
| ` like` | -0.8166 | 44.2% |
| ` the` | -0.3461 | 70.7% |
| ` last` | -1.5937 | 20.3% |
| ` digit` | -1.2979 | 27.3% |
| ` of` | -0.4838 | 61.6% |
| ` pi` | -0.2839 | 75.3% |
| `—` | -0.4669 | 62.7% |
| `inf` | -1.3530 | 25.8% |
| `initely` | -0.0298 | 97.1% |
| ` complex` | -0.7981 | 45.0% |
| ` yet` | -0.2137 | 80.8% |
| ` tantal` | -1.6926 | 18.4% |
| `izing` | -0.0000 | 100.0% |
| `ly` | -0.0018 | 99.8% |
| ` elusive` | -0.3984 | 67.1% |
| `,` | -0.4927 | 61.1% |
| ` leaving` | -0.4493 | 63.8% |
| ` a` | -0.9235 | 39.7% |
| ` trace` | -1.2360 | 29.1% |
| ` of` | -0.0833 | 92.0% |
| ` warmth` | -1.1863 | 30.5% |
| ` that` | -0.9808 | 37.5% |
| ` feels` | -1.5513 | 21.2% |
| ` like` | -0.3176 | 72.8% |
| ` the` | -0.1818 | 83.4% |
| ` sun` | -1.1630 | 31.3% |
| `’s` | -0.8636 | 42.2% |
| ` final` | -0.4807 | 61.8% |
| ` embrace` | -0.7702 | 46.3% |
| ` before` | -0.0960 | 90.8% |
| ` surrender` | -0.8272 | 43.7% |
| `ing` | -0.0001 | 100.0% |
| ` to` | -0.0033 | 99.7% |
| ` the` | -0.6351 | 53.0% |
| ` night` | -0.0653 | 93.7% |
| `.` | -0.0110 | 98.9% |

**Perplexity:** `1.86`

---